# Visual Exploration of Neural Networks for Flower Classification
AE2 – Visualization Presentation

## Introduction

This notebook presents visual journey into the development of a convolutional neural network (CNN) for classifying flowers using the Oxford 102 Flower Dataset. Through these visualizations, we'll explore how a deep learning model can "see" and learn to recognize different flower species.

## Dataset Information
[Souce Link](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/) | [Labels](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/categories.html) | [Images on Drive](https://drive.google.com/drive/folders/1C_7gjRPE9claxoN5bAMajXYxz-_YqBhz?usp=sharing)

The Oxford 102 Flower Dataset was compiled by Maria-Elena Nilsback and Andrew Zisserman at the Visual Geometry Group, University of Oxford. It contains images of flowers belonging to 102 different categories common in the United Kingdom. The dataset features:

- 102 flower categories
- Each category contains between 40-258 images
- Collected from various sources including professional photographs and web searches

## Project Objectives

This project aims to:

1. Visualize the data preparation process
2. Design and implement a CNN for flower classification
3. Visualize the training process and how the model learns over time
4. See what the neural network (NN) "sees" by visualizing activations and features
5. Analyze the model's performance

---
## Data Loading and Exploration

Before building the CNN, we need to load and understand the dataset:

1. Loading the image files and their labels
2. Exploring the dataset
3. Setting up data preprocessing for neural network training

We'll start by importing the libraries and loading the dataset and setting some constants

In [70]:
import torch
import time
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import scipy.io
import json
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from plotly.subplots import make_subplots


# Setting the seeds for reproducability
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

### Data Loading

In [71]:
# PyTorch device, (P.S. You're cooked ☠️ if you dont have a NVDIA GPU for this. RIP 🪦)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Directories
data_dir = 'data/102flowers/jpg'  # Directory containing the flower images
labels_file = 'data/imagelabels.mat'  # File with image labels

# Load the category-to-names map
cat_to_name = 'data/cat_to_name.json'
with open(cat_to_name, 'r') as f:
    categories_dir = json.load(f)

categories = list(categories_dir.values())

Using device: cuda


Labels is in order the first label (index 0) corresponds to image_00001

Here lets link each to them and the names category; image name format is image_00001.jpg to image_08189.jpg, then split the data before we start visually looking

In [72]:
labels = scipy.io.loadmat(labels_file)['labels'][0]
# str(i).zfill(5) pads with 0's for 5 total digits
image_names = [f'image_{str(i).zfill(5)}.jpg' for i in range(1, len(labels)+1)]

In [73]:
df = pd.DataFrame(zip(labels, image_names), columns=['label', 'image'])
df['flower_name'] = df['label'].map(lambda x: categories_dir.get(str(x), 'Unknown'))
df.sample(2)

,label,image,flower_name
2126,75,image_02127.jpg,thorn apple
4270,18,image_04271.jpg,peruvian lily


In [74]:
# Train Test Val split the indexes at 60 / 20 / 20
train_idx, temp_idx = train_test_split(df, test_size=0.4, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

print(f"Number of training images:      {len(train_idx)}")
print(f"Number of validation images:    {len(val_idx)}")
print(f"Number of test images:          {len(test_idx)}")
print(f"Total number of images:         {len(labels)}")
print(f"Number of classes:              {len(np.unique(labels))}")

Number of training images:      4913
Number of validation images:    1638
Number of test images:          1638
Total number of images:         8189
Number of classes:              102


### Dataset Exploration

I picked the dataset due to its diffucult nature with many flower looking similar, with light differences that even humans might not be able to always recognize. The dataset includes variations in lighting, angle, and background, making it an great test for a CNN's ability to learn meaningful features.

This project wont go into too much detail on the flowers dataset itself but rather the Deep Learning CNN for it; however its still important to understand the dataset at a surface level first.

#### Distribution Analysis (bar chart)

In [75]:
class_counts = df['label'].value_counts().sort_index()

# Class Distribution with color
fig = px.bar(
    x=categories,
    y=class_counts.values,
    color=class_counts.values,
    labels={'x': 'Flower Name', 'y': 'Number of Images'},
    title='Distribution of Flower Classes in Dataset',
    color_continuous_scale='Viridis' 
)
fig.update_layout(
    xaxis=dict(
        tickangle=90,
        tickfont=dict(size=8)
    ),
)
fig.show()

print(f"Average images / class:   {class_counts.mean():.2f}")
print(f"Minimum images / class:   {class_counts.min()}")
print(f"Maximum images / class:   {class_counts.max()}")


Average images / class:   80.28
Minimum images / class:   40
Maximum images / class:   258


The distribution of the classes shows an imbalance. The average class contains approximately 80 images, but there's a wide difference between the least represented flowers (40 images) and the most common ones (258 images), of 645%.

This does might actually cause some issues later down the line during the training process. With uneven representation the model will naturally learn from more common types during the iterations, which can lead to classfication bias where a model becomes great at some classes while underperforming in other ones.

Although in some specific use cases this might be an advantage, for exmaple if more common flowers, are ones that will be scanned more using the model post production. Although its a good idea to keep this in mind for when we analyze the model to then explore potential augmentation methods to get more data from the under-represented ones. I'll keep the imbalaneced dataset as is, unless It causes issues with the models later on.

#### Dimention Analysis (histogram + boxplot)

In [76]:
def get_dimentions(row):
    '''Get the width and height of an image'''
    img_path = os.path.join(data_dir, row['image'])     # Find file
    img = Image.open(img_path)                          # Open the image
    return img.size                                     # (width, height)

# Sample for simplicity and to get a general overview
sample = 200
dimentions  = [get_dimentions(df.iloc[n]) for n in range(0,sample)]

dimension_df = pd.DataFrame(dimentions, columns=["height", "width"])
# combine into width height into 1 col with a catergorical column
long_df = dimension_df.melt(var_name="dimention")

# Histogram and boxplot of spreads
px.histogram(long_df, x="value", color="dimention", marginal="box")

Keeping up with the distributions, here is an assesment of the dimentions spread (height, width), this shows a lot of variability in the dataset. The plot shows that most images maintain relatively similar aspect ratios with most of the widhts and heights being on one column each, but there's still a variation in overall image size with many that cont conform the majority size.

We can see outliers that are one of; unusually small or large.

This disparity in size does inpact the design of the Neural Network:

1. **Input standardization**: All images need to be of common dimension (ex. 200x200)
2. **Information loss**: While It's necessary to not overcomplicate the the NN, cropping or stretching images will cause loss of detail
3. **Aspect ratio changes**: Converting images to square dimensions will slightly distort some flowers

This will have to be considered as one of the steps for the preprocessing pipeline. 

#### Color Channel Arrays (imshow)

In [77]:
def img_to_array(row):
    '''Function to convert a specific flower image to an array given a row.
    Ex usage: img_to_array(df.iloc[0])'''
    
    img_path = os.path.join(data_dir, row['image'])
    img = Image.open(img_path)                       
    return np.array(img)                            # Convert to array

def img_to_channel(row, channel=0):
    '''Function to extract a specific channel from the image array.
    channel: 0=R, 1=G, 2=B for RGB images'''
    
    array = img_to_array(row)
    
    # Filter to only keep specific channel data
    return np.flipud(array[:, :, channel])          # extracting channel flips image, this flips it back

# image to decompose
img_id = 0

# Extract image data
img_main = px.imshow(img_to_array(df.iloc[img_id]))
img_red = px.imshow(img_to_channel(df.iloc[img_id], channel=0))
img_green = px.imshow(img_to_channel(df.iloc[img_id], channel=1))
img_blue = px.imshow(img_to_channel(df.iloc[img_id], channel=2))

# Putting them on the same graph
fig = make_subplots(rows=1, cols=4, subplot_titles=["Original", "Red", "Green", "Blue"])
fig.add_trace(img_main.data[0], row=1, col=1)
fig.add_trace(img_red.data[0], row=1, col=2)
fig.add_trace(img_green.data[0], row=1, col=3)
fig.add_trace(img_blue.data[0], row=1, col=4)

fig.update_layout(title_text="Image Channels")
fig.show()

This breaks down a flower image, controlled by the `img_id`, into its fundamental RGB color channels, with intensity shown by the heatmap overlayes on top; this shows how the neural network actually "sees" the image.

While we are capable of understanding the left image (Original) computers only work with numerical arrays representing the intensity of pixels, usually using the Red Green and Blue chells. These combines create full colors. Thes brighter areas indicate higher values on that picel are for that specific channel.

This is one of the primary porblems with computer vision as NNs don't inherently see or understand patterns like the colors, petals or shapes. They look at the raw numerical values and need to learn to identify patterns. The classification CNN will need to figure these out by itself using the arrays of data that we extract from the images.

#### Color Pattern Analysis (k-means)
The reason I've decided to use K-means clustering here over traditional R G B addition then finding the top, is due to how K-means looks for color groupings and intensity in a multi-dimensional space (RGB) and also intesity, by combining RGB so its not only those 3 values; all which simple color values aggregations can't replicate.

In [78]:
from sklearn.utils import shuffle

def extract_colors(row, n_colors=5):
    """Extract the dominant colors from an image using K-means"""

    # Load image and reshape to be a list of pixels
    img_path = os.path.join(data_dir, row['image'])
    img = Image.open(img_path)   
    img_array = np.array(img)
    h, w, c = img_array.shape
    reshaped_img = img_array.reshape(h * w, c)
    
    # Sample of pixels to speed it up
    sample = shuffle(reshaped_img)[:10000]
    
    # Fit K-means, and get top n_colors
    kmeans = KMeans(n_clusters=n_colors)
    kmeans.fit(sample)
    
    colors = kmeans.cluster_centers_            # Get the colors
    colors = colors.astype(int)                 # Converting to integer RGB values
    labels = kmeans.predict(reshaped_img)       # Labels for each pixel
    counts = np.bincount(labels)                # Percentage of each color
    percentages = counts / len(labels) * 100
    
    return colors, percentages

Each color gives 5 most promenant colors in the image along side the percetage of its appearance.

In [79]:
# Extract colors and percentages for n images

clusters = []
for i in range(50):
    colors, percentages = extract_colors(df.iloc[i])

    # RGB values in dataframe along side the Percetage
    for color, percentage in zip(colors, percentages):
        clusters.append({
            'R': color[0],
            'G': color[1],
            'B': color[2],
            'Percentage': percentage,
            'Image': df.iloc[i]['image']
        })
clusters_df = pd.DataFrame(clusters)

# RGB columns into a single color column for color
clusters_df['color'] = clusters_df.apply(lambda row: f"rgb({row['R']}, {row['G']}, {row['B']})", axis=1)

# 3D scatter plot
fig = px.scatter_3d(
    clusters_df,
    x='R',
    y='G',
    z='B',
    size='Percentage',
    hover_data=['Image'],
    title='3D Scatter Plot of RGB Values and Percentages',
    height=600
)
# Set RGB color for each point
fig.update_traces(marker=dict(
    color=clusters_df['color'],  # Use the 'color' column for RGB values
    opacity=0.8,
))

fig.show()

The K-means clustering shows the most promenant colors in the images within the dataset; including hues and tones along side the hierarchy (percetange) of the appearance from the size:

1. **Green tones** are the most common, these mostly represent the leaves and stems which are almost always green
2. **Flower-specific colors** moving slighlty further away from the greens one can see pinks, purples, blues, and white being well-represented
3. **Background colors** colors such as sky blue and browns are also present may also represent the background

This color chart is important to understand for the neural network, as color is one of the primary distinguishing feature for many flower species. The model will need to be able to use these color patterns to its advantage to classify the flowers.

#### Data Analysis -> CNN Design
This section has been exploration of the Oxford 102 Flowers Dataset's attributes trough visualizing:
* Distribution of classes
* Image dimentions
* RGB channels
* Color petterns

For the next section, there will be more of a fucus on a subset of the categories to showcase a NN building process. I;ll go over key concepts and visualization techniques while also keeping inmind constraits like training time and GPU VRAM limitations.

---

## Convolutuion Neural Network Design